In [1]:
# Install scikit-learn into the environment used by this notebook.
# Scikit-learn provides the dataset generator, train-test split,
# regression model and evaluation metrics.
%pip install scikit-learn

   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 8.3/8.3 MB 85.8 MB/s  0:00:00

   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   ---------------- ----------------------- 2/5 [cloudpickle]
   ------------------------ --------------- 3/5 [joblib]
   ------------------------ --------------- 3/5 [joblib]
   ------------------------ --------------- 3/5 [joblib]
   -------------------------------- ------- 4/5 [scikit-learn]
   -------------------------------- ------- 4/5 [scikit-learn]
   -------------------------------- ------- 4/5 [scikit-learn]
   -------------------------------- ------- 4/5 [scikit-learn]
   --------------

# Week 5: Multi-Output Regression for FiveM Server Performance

## Project Scenario

A FiveM server generates several operational measurements, including player count, active resources, script events, database queries and network latency.

In this exercise, we will train a regression model to predict three server-performance measurements at the same time:

1. CPU usage percentage
2. Memory usage in gigabytes
3. Server tick time in milliseconds

This is a **multi-output regression** problem because the model receives several input features and predicts more than one target variable.

The dataset used in this notebook is synthetic. It represents a realistic FiveM-style scenario without using private data from a real production server.

In [1]:
# Import NumPy for generating numerical data
# and performing mathematical operations.
import numpy as np

# Import pandas for storing the server data
# in a labelled table called a DataFrame.
import pandas as pd

# Import Matplotlib for creating graphs.
import matplotlib.pyplot as plt

# Import Seaborn for statistical visualisations.
import seaborn as sns

# Import the function used to divide our dataset
# into separate training and testing subsets.
from sklearn.model_selection import train_test_split

# Import the multiple linear regression model.
from sklearn.linear_model import LinearRegression

# Import regression evaluation metrics.
#
# R² measures how much variation the model explains.
# MAE measures the average absolute prediction error.
# MSE measures the average squared prediction error.
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

# Set a consistent visual style for every Seaborn graph.
sns.set_theme(style="whitegrid")

# Display confirmation that the imports succeeded.
print("All Week 5 libraries imported successfully.")

All Week 5 libraries imported successfully.


## Synthetic FiveM Server Dataset

We will generate 500 server-monitoring observations.

Each row represents one observation of the server at a particular time. The values are generated within realistic ranges and contain random variation to represent factors that our five selected features do not capture.

In [2]:
# Create a modern NumPy random-number generator.
#
# The value 42 is the random seed. Using the same seed
# means that the notebook generates the same dataset
# whenever it runs.
rng = np.random.default_rng(seed=42)

# Set the number of server observations to generate.
number_of_samples = 500

# Generate a realistic number of connected players.
# The upper boundary is excluded, so the possible values
# range from 10 to 200.
concurrent_players = rng.integers(
    low=10,
    high=201,
    size=number_of_samples
)

# Generate the number of active FiveM resources.
#
# These could include jobs, inventories, vehicles,
# user interfaces and administrative systems.
active_resources = rng.integers(
    low=25,
    high=151,
    size=number_of_samples
)

# Generate the approximate number of script events
# processed by the server during one minute.
script_events_per_minute = rng.integers(
    low=100,
    high=1501,
    size=number_of_samples
)

# Generate the number of database queries per minute.
#
# Database activity could come from inventories,
# player saves, banking, housing and vehicle systems.
database_queries_per_minute = rng.integers(
    low=50,
    high=801,
    size=number_of_samples
)

# Generate average player latency in milliseconds.
average_latency_ms = rng.uniform(
    low=20,
    high=120,
    size=number_of_samples
)

# Generate random unexplained variation for CPU usage.
#
# A real server's performance is affected by many factors
# that are not included in this small educational dataset.
cpu_noise = rng.normal(
    loc=0,
    scale=3,
    size=number_of_samples
)

# Generate random variation for memory usage.
memory_noise = rng.normal(
    loc=0,
    scale=0.4,
    size=number_of_samples
)

# Generate random variation for server tick time.
tick_noise = rng.normal(
    loc=0,
    scale=0.8,
    size=number_of_samples
)

# Create synthetic CPU usage.
#
# Each coefficient describes how strongly a feature
# contributes to the simulated CPU load.
cpu_usage_percent = (
    8
    + (0.16 * concurrent_players)
    + (0.09 * active_resources)
    + (0.012 * script_events_per_minute)
    + (0.008 * database_queries_per_minute)
    + cpu_noise
)

# Limit CPU usage to a valid range from 0% to 100%.
cpu_usage_percent = np.clip(
    cpu_usage_percent,
    0,
    100
)

# Create synthetic memory usage in gigabytes.
#
# Player sessions and active resources have a strong
# influence on the simulated memory requirement.
memory_usage_gb = (
    1.5
    + (0.022 * concurrent_players)
    + (0.038 * active_resources)
    + (0.0015 * database_queries_per_minute)
    + memory_noise
)

# Prevent impossible negative memory values.
memory_usage_gb = np.clip(
    memory_usage_gb,
    0,
    None
)

# Create synthetic server tick time in milliseconds.
#
# Higher tick time can indicate that the server requires
# longer to process each update cycle.
server_tick_time_ms = (
    1.0
    + (0.012 * concurrent_players)
    + (0.003 * script_events_per_minute)
    + (0.004 * database_queries_per_minute)
    + (0.020 * average_latency_ms)
    + tick_noise
)

# Prevent impossible negative tick-time values.
server_tick_time_ms = np.clip(
    server_tick_time_ms,
    0,
    None
)

# Combine the generated arrays into a labelled DataFrame.
server_data = pd.DataFrame({
    "ConcurrentPlayers": concurrent_players,
    "ActiveResources": active_resources,
    "ScriptEventsPerMinute": script_events_per_minute,
    "DatabaseQueriesPerMinute": database_queries_per_minute,
    "AverageLatencyMs": average_latency_ms,
    "CPUUsagePercent": cpu_usage_percent,
    "MemoryUsageGB": memory_usage_gb,
    "ServerTickTimeMs": server_tick_time_ms
})

# Display the first five server observations.
server_data.head()

,ConcurrentPlayers,ActiveResources,ScriptEventsPerMinute,DatabaseQueriesPerMinute,AverageLatencyMs,CPUUsagePercent,MemoryUsageGB,ServerTickTimeMs
0,27,39,1232,447,26.206311,37.078882,4.312228,6.324596
1,157,107,1132,245,65.826204,56.120167,9.131304,7.525990
2,135,146,882,221,32.903006,52.530722,10.241143,6.684372
3,93,134,1341,292,35.232671,54.747118,8.981548,7.060549
4,92,59,279,498,83.228281,38.327660,6.219245,6.035362


## Data Exploration and Validation

Before training the regression model, we will check:

- The number of rows and columns
- The data type of each column
- Whether any values are missing
- Whether duplicate observations exist
- The minimum, maximum, mean and standard deviation
- Whether the generated values fall within sensible ranges

This stage helps identify data-quality problems before they affect the model.

In [3]:
# Display the number of rows and columns.
#
# We expect 500 observations and 8 variables.
print("Dataset shape:", server_data.shape)

# Display the column names.
print("\nDataset columns:")
for column in server_data.columns:
    print("-", column)

# Display a concise description of the DataFrame.
#
# This shows each column's data type and whether
# every row contains a non-null value.
print("\nDataset information:")
server_data.info()

Dataset shape: (500, 8)

Dataset columns:
- ConcurrentPlayers
- ActiveResources
- ScriptEventsPerMinute
- DatabaseQueriesPerMinute
- AverageLatencyMs
- CPUUsagePercent
- MemoryUsageGB
- ServerTickTimeMs

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ConcurrentPlayers         500 non-null    int64  
 1   ActiveResources           500 non-null    int64  
 2   ScriptEventsPerMinute     500 non-null    int64  
 3   DatabaseQueriesPerMinute  500 non-null    int64  
 4   AverageLatencyMs          500 non-null    float64
 5   CPUUsagePercent           500 non-null    float64
 6   MemoryUsageGB             500 non-null    float64
 7   ServerTickTimeMs          500 non-null    float64
dtypes: float64(4), int64(4)
memory usage: 31.4 KB


In [4]:
# Count missing values in every column.
missing_values = server_data.isna().sum()

print("Missing values per column:")
print(missing_values)

# Count rows that are exact duplicates.
duplicate_rows = server_data.duplicated().sum()

print("\nNumber of duplicate rows:", duplicate_rows)

# Confirm whether the dataset passed both checks.
if missing_values.sum() == 0 and duplicate_rows == 0:
    print("\nData-quality check passed.")
else:
    print("\nThe dataset requires cleaning.")

Missing values per column:
ConcurrentPlayers           0
ActiveResources             0
ScriptEventsPerMinute       0
DatabaseQueriesPerMinute    0
AverageLatencyMs            0
CPUUsagePercent             0
MemoryUsageGB               0
ServerTickTimeMs            0
dtype: int64

Number of duplicate rows: 0

Data-quality check passed.


In [5]:
# Calculate descriptive statistics for every numerical column.
#
# count = number of observations
# mean  = average value
# std   = standard deviation
# min   = smallest value
# 25%   = first quartile
# 50%   = median
# 75%   = third quartile
# max   = largest value
summary_statistics = server_data.describe().T

# Round the results to two decimal places
# so the table is easier to read.
summary_statistics = summary_statistics.round(2)

# Display the transposed statistics table.
summary_statistics

,count,mean,std,min,25%,50%,75%,max
ConcurrentPlayers,500.0,104.84,54.62,11.00,59.75,104.00,153.00,200.00
ActiveResources,500.0,88.36,36.73,25.00,55.00,90.00,121.25,150.00
ScriptEventsPerMinute,500.0,811.45,406.35,101.00,485.00,804.50,1168.00,1499.00
DatabaseQueriesPerMinute,500.0,413.99,219.98,51.00,222.50,408.00,607.00,797.00
AverageLatencyMs,500.0,70.46,29.15,20.05,44.61,69.25,95.91,119.94
CPUUsagePercent,500.0,45.25,10.70,16.05,37.38,44.94,52.63,75.34
MemoryUsageGB,500.0,7.79,1.87,3.15,6.57,7.72,9.11,12.70
ServerTickTimeMs,500.0,7.80,1.92,2.53,6.47,7.87,9.15,12.89


## Input Features and Target Variables

The model receives five input features describing the current FiveM server workload.

It predicts three target variables describing server performance.

In machine-learning notation:

- `X` contains the input features.
- `Y` contains the target outputs.

Capital `Y` is useful here because we have a matrix containing three output columns rather than one target vector.

In [6]:
# List the columns that the model will use as inputs.
feature_columns = [
    "ConcurrentPlayers",
    "ActiveResources",
    "ScriptEventsPerMinute",
    "DatabaseQueriesPerMinute",
    "AverageLatencyMs"
]

# List the three performance measurements
# that the model will learn to predict.
target_columns = [
    "CPUUsagePercent",
    "MemoryUsageGB",
    "ServerTickTimeMs"
]

# Create the input-feature matrix.
X = server_data[feature_columns]

# Create the multi-output target matrix.
Y = server_data[target_columns]

# Display the dimensions of both matrices.
print("Input matrix shape:", X.shape)
print("Target matrix shape:", Y.shape)

# Display the first five input rows.
print("\nInput features:")
display(X.head())

# Display the corresponding target values.
print("Target variables:")
display(Y.head())

Input matrix shape: (500, 5)
Target matrix shape: (500, 3)

Input features:


,ConcurrentPlayers,ActiveResources,ScriptEventsPerMinute,DatabaseQueriesPerMinute,AverageLatencyMs
0,27,39,1232,447,26.206311
1,157,107,1132,245,65.826204
2,135,146,882,221,32.903006
3,93,134,1341,292,35.232671
4,92,59,279,498,83.228281


Target variables:


,CPUUsagePercent,MemoryUsageGB,ServerTickTimeMs
0,37.078882,4.312228,6.324596
1,56.120167,9.131304,7.525990
2,52.530722,10.241143,6.684372
3,54.747118,8.981548,7.060549
4,38.327660,6.219245,6.035362
